# ProQuest TDM Studio - NYT XML Parser (Updated)

## Project: News Framing Analysis
**Dataset:** New York Times Articles (2005-2025)  
**Updated:** December 2025  
**Objective:** Parse XML files to extract article content, metadata, and document types for framing analysis

---

## Updates from Previous Version:
1. **No document type filtering** - Load all document types from TDM Studio
2. **Extract document type from XML** - Identify articles, editorials, obituaries, etc.
3. **JSON export format** - Use JSON instead of CSV to prevent data corruption
4. **Timestamped filenames** - Include date in output filenames

---

## 1. Setup and Imports

Import all necessary libraries for XML parsing, data manipulation, and file handling.

In [ ]:
# Standard library imports
import os
import glob
from pathlib import Path
import json
from datetime import datetime
import re

# XML parsing
import xml.etree.ElementTree as ET
from xml.etree.ElementTree import ParseError

# Data manipulation and analysis
import pandas as pd
import numpy as np

# Progress tracking
from tqdm import tqdm

# Display settings for better readability
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)

print("✓ All libraries imported successfully")

## 2. Configuration

Set up paths and parameters for data processing.

In [ ]:
# Configuration parameters
NEWSPAPER_NAME = "new_york_times"  # Used for file naming
DATA_DIR = Path('data/new_york_time')  # Path to NYT dataset
OUTPUT_DIR = Path('results')  # Output directory
SAMPLE_SIZE = 100  # Number of articles to process (set to None for all articles)

# Create output directory if it doesn't exist
OUTPUT_DIR.mkdir(exist_ok=True)

# Generate timestamp for file naming (format: YYYYMMDD)
DATE_STAMP = datetime.now().strftime('%Y%m%d')

print(f"Configuration:")
print(f"  Newspaper: {NEWSPAPER_NAME}")
print(f"  Data directory: {DATA_DIR}")
print(f"  Output directory: {OUTPUT_DIR}")
print(f"  Sample size: {SAMPLE_SIZE if SAMPLE_SIZE else 'All articles'}")
print(f"  Date stamp: {DATE_STAMP}")

## 3. Data Exploration

Explore the data directory structure and count available XML files.

In [ ]:
# Check if the directory exists
if DATA_DIR.exists():
    print(f"✓ Data directory found: {DATA_DIR}")
else:
    print(f"✗ Data directory not found: {DATA_DIR}")
    print("Please check the path and update DATA_DIR variable.")

In [ ]:
# Find all XML files in the directory
xml_files = list(DATA_DIR.glob('**/*.xml'))

print(f"Total XML files found: {len(xml_files):,}")

# Display first few file paths
if xml_files:
    print(f"\nFirst 5 XML files:")
    for i, file_path in enumerate(xml_files[:5], 1):
        print(f"{i}. {file_path.name}")
else:
    print("No XML files found. Please check the directory structure.")

## 4. XML Structure Inspection

Before parsing, let's examine the XML structure to identify available fields, especially the document type field.

In [ ]:
def inspect_xml_structure(xml_file_path, show_content=False):
    """
    Inspect the structure of an XML file to understand its schema.
    
    Parameters:
    -----------
    xml_file_path : Path or str
        Path to the XML file to inspect
    show_content : bool
        If True, display sample content from each tag
    """
    try:
        tree = ET.parse(xml_file_path)
        root = tree.getroot()
        
        print(f"Root tag: {root.tag}")
        print(f"Root attributes: {root.attrib}")
        print("\n" + "="*80)
        print("All unique tags in the XML:")
        print("="*80)
        
        # Collect all tags with sample content
        tag_info = {}
        for elem in root.iter():
            if elem.tag not in tag_info:
                # Store first non-empty text content for each tag
                content = elem.text.strip() if elem.text else None
                tag_info[elem.tag] = content
        
        # Display tags, highlighting potential document type fields
        doc_type_keywords = ['type', 'document', 'object', 'publication', 'content', 'category']
        
        for tag in sorted(tag_info.keys()):
            # Check if this might be a document type field
            is_potential_doctype = any(keyword in tag.lower() for keyword in doc_type_keywords)
            marker = " ⭐ [POTENTIAL DOCUMENT TYPE FIELD]" if is_potential_doctype else ""
            
            if show_content and tag_info[tag]:
                # Show first 50 characters of content
                content_preview = tag_info[tag][:50] + "..." if len(tag_info[tag]) > 50 else tag_info[tag]
                print(f"  - {tag}: \"{content_preview}\"{marker}")
            else:
                print(f"  - {tag}{marker}")
        
        print("\n" + "="*80)
        print("Note: Tags marked with ⭐ might contain document type information")
        print("="*80)
        
    except Exception as e:
        print(f"Error inspecting XML: {e}")

# Inspect the first XML file
if xml_files:
    print(f"Inspecting: {xml_files[0].name}\n")
    inspect_xml_structure(xml_files[0], show_content=True)

## 5. XML Parsing Functions

Updated parsing function that extracts document type and other metadata.

In [ ]:
def parse_single_xml(xml_file_path):
    """
    Parse a single XML file and extract article information including document type.
    
    Parameters:
    -----------
    xml_file_path : Path or str
        Path to the XML file
    
    Returns:
    --------
    dict : Dictionary containing extracted article data
           Returns None if parsing fails
    """
    try:
        tree = ET.parse(xml_file_path)
        root = tree.getroot()
        
        # Initialize article data dictionary
        article_data = {
            'file_name': Path(xml_file_path).name,
            'document_id': Path(xml_file_path).stem,  # File name without extension
            'document_type': None,  # ⭐ NEW: Document type field
            'title': None,
            'author': None,
            'publication_date': None,
            'publication_date_numeric': None,
            'section': None,
            'word_count': None,
            'abstract': None,
            'full_text': None,
            'subjects': [],
            'locations': [],
            'people': [],
            'companies': [],
            'source_type': None,
            'publication_title': None,
            'url': None
        }
        
        # ⭐ CRITICAL: Extract document type
        # Try multiple possible tag names for document type
        doc_type_tags = [
            './/ObjectType',
            './/DocumentType', 
            './/PublicationType',
            './/ContentType',
            './/Type',
            './/DocType'
        ]
        
        for tag in doc_type_tags:
            elem = root.find(tag)
            if elem is not None and elem.text:
                article_data['document_type'] = elem.text.strip()
                break
        
        # Extract publication title
        pub_title_elem = root.find('.//PublicationTitle')
        if pub_title_elem is not None:
            article_data['publication_title'] = pub_title_elem.text
        
        # Extract source type
        source_elem = root.find('.//SourceType')
        if source_elem is not None:
            article_data['source_type'] = source_elem.text
        
        # Extract title
        title_tags = ['.//Title', './/TitleAtt', './/ArticleTitle']
        for tag in title_tags:
            elem = root.find(tag)
            if elem is not None and elem.text:
                article_data['title'] = elem.text.strip()
                break
        
        # Extract publication dates
        date_elem = root.find('.//NumericDate')
        if date_elem is not None:
            article_data['publication_date_numeric'] = date_elem.text
        
        alpha_date_elem = root.find('.//AlphaDate')
        if alpha_date_elem is not None:
            article_data['publication_date'] = alpha_date_elem.text
        
        # Extract author(s)
        author_tags = ['.//Author', './/Byline', './/Creator']
        for tag in author_tags:
            elem = root.find(tag)
            if elem is not None and elem.text:
                article_data['author'] = elem.text.strip()
                break
        
        # Extract section/category
        section_elem = root.find('.//Section')
        if section_elem is not None:
            article_data['section'] = section_elem.text
        
        # Extract word count
        wc_elem = root.find('.//WordCount')
        if wc_elem is not None:
            try:
                article_data['word_count'] = int(wc_elem.text)
            except (ValueError, TypeError):
                article_data['word_count'] = None
        
        # Extract abstract/summary
        abstract_elem = root.find('.//Abstract')
        if abstract_elem is not None:
            article_data['abstract'] = abstract_elem.text
        
        # Extract full text - try multiple possible tags
        text_tags = ['.//FullText', './/Text', './/AbsText', './/BodyText']
        for tag in text_tags:
            elem = root.find(tag)
            if elem is not None and elem.text:
                article_data['full_text'] = elem.text.strip()
                break
        
        # Extract subjects/topics
        subject_elems = root.findall('.//Subject')
        article_data['subjects'] = [s.text.strip() for s in subject_elems if s.text]
        
        # Extract locations
        location_elems = root.findall('.//Location')
        article_data['locations'] = [loc.text.strip() for loc in location_elems if loc.text]
        
        # Extract people mentioned
        people_elems = root.findall('.//Person')
        article_data['people'] = [p.text.strip() for p in people_elems if p.text]
        
        # Extract companies mentioned
        company_elems = root.findall('.//Company')
        article_data['companies'] = [c.text.strip() for c in company_elems if c.text]
        
        # Extract URL
        url_elem = root.find('.//URL')
        if url_elem is not None:
            article_data['url'] = url_elem.text
        
        return article_data
    
    except ParseError as e:
        print(f"XML Parse Error in {xml_file_path}: {e}")
        return None
    except Exception as e:
        print(f"Error processing {xml_file_path}: {e}")
        return None

print("✓ XML parsing function defined successfully")

## 6. Process Articles

Parse XML files and extract data. Processing a sample first to validate the parsing logic.

In [ ]:
# Determine how many files to process
if SAMPLE_SIZE and SAMPLE_SIZE < len(xml_files):
    sample_files = xml_files[:SAMPLE_SIZE]
    print(f"Processing sample of {len(sample_files)} articles...\n")
else:
    sample_files = xml_files
    print(f"Processing all {len(sample_files):,} articles...\n")

# Parse all XML files
parsed_articles = []
failed_files = []

# Use tqdm for progress tracking
for xml_file in tqdm(sample_files, desc="Parsing XML files"):
    article_data = parse_single_xml(xml_file)
    
    if article_data is not None:
        parsed_articles.append(article_data)
    else:
        failed_files.append(xml_file.name)

# Summary statistics
print(f"\n{'='*80}")
print(f"PARSING SUMMARY")
print(f"{'='*80}")
print(f"✓ Successfully parsed: {len(parsed_articles):,} articles")
print(f"✗ Failed to parse: {len(failed_files):,} articles")
print(f"Success rate: {len(parsed_articles)/len(sample_files)*100:.1f}%")

if failed_files:
    print(f"\nFirst 10 failed files:")
    for fname in failed_files[:10]:
        print(f"  - {fname}")

## 7. Convert to DataFrame and Analyze

Convert parsed data to pandas DataFrame for analysis.

In [ ]:
# Create DataFrame
df = pd.DataFrame(parsed_articles)

print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

# Display first few rows
print("\nFirst 3 articles:")
display(df.head(3))

### Data Quality Assessment

In [ ]:
print("="*80)
print("DATA QUALITY ASSESSMENT")
print("="*80)

# Check completeness of key fields
key_fields = ['document_type', 'title', 'full_text', 'publication_date', 'author']

print("\nField completeness:")
for field in key_fields:
    if field in df.columns:
        count = df[field].notna().sum()
        percentage = count / len(df) * 100
        print(f"  {field:25s}: {count:6,} ({percentage:5.1f}%)")

# ⭐ IMPORTANT: Analyze document types
print("\n" + "="*80)
print("DOCUMENT TYPE DISTRIBUTION")
print("="*80)

if 'document_type' in df.columns and df['document_type'].notna().any():
    doc_type_counts = df['document_type'].value_counts()
    print(f"\nFound {len(doc_type_counts)} different document types:\n")
    for doc_type, count in doc_type_counts.items():
        percentage = count / len(df) * 100
        print(f"  {doc_type:30s}: {count:6,} ({percentage:5.1f}%)")
else:
    print("\n⚠️  WARNING: No document type information found!")
    print("   Please check the XML structure inspection above to identify the correct tag.")

# Date range
if 'publication_date' in df.columns and df['publication_date'].notna().any():
    print(f"\nDate range:")
    print(f"  Earliest: {df['publication_date'].min()}")
    print(f"  Latest: {df['publication_date'].max()}")

### Display Sample Article

In [ ]:
# Display a complete sample article
print("="*80)
print("SAMPLE ARTICLE")
print("="*80)

if len(df) > 0:
    # Try to get an article with full text
    sample_idx = df[df['full_text'].notna()].index[0] if df['full_text'].notna().any() else 0
    sample = df.iloc[sample_idx]
    
    for key, value in sample.items():
        if key == 'full_text' and value:
            # Show only first 300 characters of full text
            print(f"\n{key}:")
            print(str(value)[:300] + "..." if len(str(value)) > 300 else value)
        elif isinstance(value, list) and value:
            print(f"\n{key}: {', '.join(str(v) for v in value[:5])}")
        else:
            print(f"\n{key}: {value}")

## 8. Export Data to JSON

Export parsed data to JSON format with timestamped filename.
**Note:** Using JSON instead of CSV as recommended by advisor to prevent data corruption.

In [ ]:
# Generate output filename with date stamp
output_filename = f"{NEWSPAPER_NAME}_{DATE_STAMP}.json"
output_path = OUTPUT_DIR / output_filename

print(f"Exporting data to JSON format...\n")

# Export to JSON
# Using orient='records' to create a list of dictionaries
# indent=2 for readable formatting
# force_ascii=False to preserve non-ASCII characters
df.to_json(
    output_path, 
    orient='records', 
    indent=2, 
    force_ascii=False
)

# Get file size
file_size_mb = output_path.stat().st_size / (1024 * 1024)

print(f"✓ JSON exported successfully!")
print(f"  File: {output_path}")
print(f"  Size: {file_size_mb:.2f} MB")
print(f"  Records: {len(df):,}")

### Optional: Also Export to Pickle

Pickle format is useful for quickly loading data back into Python while preserving all data types.

In [ ]:
# Export to Pickle (optional, but useful for Python)
pickle_filename = f"{NEWSPAPER_NAME}_{DATE_STAMP}.pkl"
pickle_path = OUTPUT_DIR / pickle_filename

df.to_pickle(pickle_path)

pickle_size_mb = pickle_path.stat().st_size / (1024 * 1024)

print(f"✓ Pickle file also exported")
print(f"  File: {pickle_path}")
print(f"  Size: {pickle_size_mb:.2f} MB")

## 9. Create Processing Summary

Generate a summary report of the processing.

In [ ]:
# Create comprehensive summary
summary = {
    'newspaper': NEWSPAPER_NAME,
    'processing_date': DATE_STAMP,
    'processing_timestamp': datetime.now().isoformat(),
    'data_directory': str(DATA_DIR),
    'total_xml_files_available': len(xml_files),
    'files_processed': len(sample_files),
    'successfully_parsed': len(parsed_articles),
    'failed_to_parse': len(failed_files),
    'success_rate': f"{len(parsed_articles)/len(sample_files)*100:.2f}%",
    'data_quality': {
        'articles_with_document_type': int(df['document_type'].notna().sum()) if 'document_type' in df.columns else 0,
        'articles_with_full_text': int(df['full_text'].notna().sum()) if 'full_text' in df.columns else 0,
        'articles_with_title': int(df['title'].notna().sum()) if 'title' in df.columns else 0,
        'articles_with_author': int(df['author'].notna().sum()) if 'author' in df.columns else 0
    },
    'document_type_distribution': df['document_type'].value_counts().to_dict() if 'document_type' in df.columns else {},
    'date_range': {
        'earliest': str(df['publication_date'].min()) if 'publication_date' in df.columns else None,
        'latest': str(df['publication_date'].max()) if 'publication_date' in df.columns else None
    },
    'output_files': {
        'json': str(output_path),
        'json_size_mb': f"{file_size_mb:.2f}",
        'pickle': str(pickle_path),
        'pickle_size_mb': f"{pickle_size_mb:.2f}"
    }
}

# Save summary as JSON
summary_filename = f"{NEWSPAPER_NAME}_{DATE_STAMP}_summary.json"
summary_path = OUTPUT_DIR / summary_filename

with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)

# Display summary
print("="*80)
print("PROCESSING SUMMARY")
print("="*80)
print(json.dumps(summary, indent=2))
print(f"\n✓ Summary saved to: {summary_path}")

---

## Next Steps

After validating the results:

1. **Check the document type distribution** above
   - Verify that document types are being correctly extracted
   - Identify which types are present (News, Editorial, Obituary, etc.)

2. **Adjust XML parsing if needed**
   - If document_type is not extracted, check the XML inspection output
   - Update the `doc_type_tags` list in `parse_single_xml()` function

3. **Process full dataset**
   - Set `SAMPLE_SIZE = None` to process all articles
   - Or increase `SAMPLE_SIZE` gradually

4. **Repeat for other newspapers**
   - Washington Post
   - LA Times  
   - Chicago Tribune
   - Simply change `NEWSPAPER_NAME` and `DATA_DIR` variables

5. **Document Type Analysis**
   - Filter articles by document type for specific analyses
   - Compare framing across different document types

---

## Important Notes

- ⭐ **Document Type Field**: Critical for distinguishing articles, editorials, obituaries, etc.
- 📁 **JSON Format**: Used instead of CSV to prevent data corruption (as advised)
- 📅 **Timestamped Files**: All output files include date for version tracking
- 🔄 **Reusable**: Same code works for all newspapers by changing configuration
- 💾 **Memory Efficient**: Process in batches if dataset is very large

---